In [ ]:
import os
from pathlib import Path
from types import SimpleNamespace

import numpy as np

try:
    from scipy import ndimage, signal, optimize, interpolate, stats
except Exception:
    ndimage = signal = optimize = interpolate = stats = None

def n_elements(x):
    if x is None:
        return 0
    try:
        return np.size(x)
    except Exception:
        return 1

def read_parameter_file(parfile):
    params = {}
    path = Path(parfile)
    if not path.exists():
        raise FileNotFoundError(parfile)

    for raw in path.read_text().splitlines():
        line = raw.split(';', 1)[0].strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()
        value = value.replace('[', 'np.array([').replace(']', '])')
        value = value.replace('^', '**')
        try:
            params[key] = eval(value, {'np': np, 'array': np.array})
        except Exception:
            params[key] = value.strip("'\"")
    return params

def write_idl_array_line(f, name, arr, comment=''):
    arr = np.asarray(arr).ravel()
    values = ','.join(f'{v:10.4E}' if abs(v) >= 1e4 or (abs(v) < 1e-3 and v != 0) else f'{v:10.4f}' for v in arr)
    f.write(f'{name.upper()}=[{values}]')
    if comment:
        f.write(f' ; {comment}')
    f.write('\n')

def robust_sigma(values):
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(values)
    return 1.4826 * np.nanmedian(np.abs(values - med))

def linear_interp(x, y, x_new):
    return np.interp(x_new, np.asarray(x, dtype=float), np.asarray(y, dtype=float))

def congrid(array, new_shape):
    array = np.asarray(array, dtype=float)
    if ndimage is None:
        raise ImportError('scipy is required for congrid')
    zoom = [n / o for n, o in zip(new_shape, array.shape)]
    return ndimage.zoom(array, zoom, order=1)

def idl_hist2d(x, y, xbin, ybin, xmin, xmax, ymin, ymax):
    x_edges = np.arange(xmin, xmax + xbin, xbin)
    y_edges = np.arange(ymin, ymax + ybin, ybin)
    hist, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    return hist

def load_model_file(path):
    with open(path, 'r') as f:
        age_line = f.readline().strip()
        feh_line = f.readline().strip()
        shape_line = f.readline().strip()
        shape = tuple(int(v) for v in shape_line.replace(',', ' ').split()[:2])
        data = np.loadtxt(f)
    return age_line, feh_line, data.reshape(shape)

def save_model_file(path, age_label, feh_label, model):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model = np.asarray(model, dtype=float)
    with open(path, 'w') as f:
        f.write(age_label + '\n')
        f.write(feh_label + '\n')
        f.write(f'{model.shape[0]} {model.shape[1]}\n')
        np.savetxt(f, model)


In [ ]:
def mass_integral(power, lo, hi):
    if np.isclose(power, 2.0):
        return np.log(hi) - np.log(lo)
    return (hi ** (2 - power) - lo ** (2 - power)) / (2 - power)

def variable_from_isochrone(iso, expression):
    words = [w.strip().lower() for w in expression.split('-')]
    values = np.asarray(getattr(iso, words[0]))
    for w in words[1:]:
        values = values - np.asarray(getattr(iso, w))
    return values

def mkmodel_art(parfile, readisoc):
    p = read_parameter_file(parfile)
    name = p.get('name', 'model')
    isoc = p.get('isoc', 'gir')
    interpol = p.get('interpol', 1)
    agebin = np.asarray(p['agebin'], dtype=float)
    agesz = np.asarray(p['agesz'], dtype=float)
    fehbin = np.asarray(p['fehbin'], dtype=float)
    imf = np.atleast_1d(np.asarray(p['imf'], dtype=float))
    mimf = np.atleast_1d(np.asarray(p.get('mimf', []), dtype=float))
    nsub = int(p.get('nsub', 1))
    fgrid = float(p.get('fgrid', 1))
    a_x = np.asarray(p.get('a_x', np.zeros(len(agebin))), dtype=float)
    a_y = np.asarray(p.get('a_y', np.zeros(len(agebin))), dtype=float)

    xmin, xmax, xbin = p['xmin'], p['xmax'], p['xbin']
    ymin, ymax, ybin = p['ymin'], p['ymax'], p['ybin']
    xvar, yvar = p['xvar'], p['yvar']

    nx = int((xmax - xmin) / xbin * fgrid + 1)
    ny = int((ymax - ymin) / ybin * fgrid + 1)
    xvec = np.arange(nx) * xbin / fgrid + xmin
    yvec = np.arange(ny) * ybin / fgrid + ymin

    for ii in range(len(agebin)):
        age_lo = agebin[ii] - agesz[ii] / 2
        age_hi = agebin[ii] + agesz[ii] / 2
        ages = np.arange(nsub) / nsub * (age_hi - age_lo) + age_lo
        fmodel = np.zeros((nx, ny), dtype=float)
        tmass = np.zeros(nsub, dtype=float)

        for kk, age in enumerate(ages):
            zbin = 10 ** fehbin[ii] * 0.019
            zmax = 0.019 if age < 10 ** (7.8 - 9.0) else 0.031
            if zbin > zmax:
                continue

            iso = readisoc(isoc, age, fehbin[ii], interpolate=bool(interpol))
            mass = np.asarray(iso.i_mass, dtype=float)
            x = variable_from_isochrone(iso, xvar) + a_x[ii]
            y = variable_from_isochrone(iso, yvar) + a_y[ii]

            dmass = mass - np.roll(mass, 1)
            if len(dmass) > 1:
                dmass[0] = dmass[1]

            imfar = np.full(len(mass), imf[-1])
            for ll in range(len(imf) - 2, -1, -1):
                if ll < len(mimf):
                    imfar[mass < mimf[ll]] = imf[ll]
            lf = mass ** (-imfar) * dmass

            use = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
            xel = ((x[use] - xmin) / xbin * fgrid).astype(int)
            yel = ((y[use] - ymin) / ybin * fgrid).astype(int)
            np.add.at(fmodel, (xel, yel), lf[use])

            if len(imf) == 1:
                tmass[kk] = mass_integral(imf[0], 0.08, 120.0)
            else:
                total = mass_integral(imf[0], 0.08, mimf[0])
                total += mass_integral(imf[-1], mimf[-1], 120.0)
                for ll in range(1, len(imf) - 1):
                    total += mass_integral(imf[ll], mimf[ll - 1], mimf[ll])
                tmass[kk] = total

        sfr_norm = np.sum(tmass) / (agesz[ii] * 1e9)
        array = fmodel / sfr_norm if sfr_norm != 0 else fmodel
        cmodel = congrid(array, (int((xmax - xmin) / xbin + 1), int((ymax - ymin) / ybin + 1)))
        norm = max(cmodel.sum(), 1.0)
        cmodel = cmodel / norm * array.sum()
        save_model_file(
            Path('models') / f'{name}{ii}.dat',
            f'Age range: {agebin[ii] - agesz[ii] / 2} - {agebin[ii] + agesz[ii] / 2} Gyr',
            f'[Fe/H]: {fehbin[ii]}',
            cmodel,
        )
